# Vectorized crossover parameter search — M6E on Tradovate (1h candles), single-symbol / single-strategy

Goal: pick **one** fast/slow crossover indicator pair and **one** parameter set to run
as M6E's only strategy in `step_2_run_full_backtest.ipynb`. This notebook is the
research pipeline that earns that choice, per
`apps/backtester/OPTIMIZATION_RESEARCH_PLAN.md`.

**Pipeline** (each stage is its own cell(s) below):
1. **Shared harness**: real Tradovate/Tradeify cost model (researched, not
   placeholders — see below) + a scorecard function computing Sharpe/Sortino/Calmar/
   max-drawdown (daily-resampled, same convention `PerformanceAnalyzer` uses) and,
   critically, a **trade-level Sharpe lower bound** (`trade_sharpe_lb`) — this is the
   ranking metric, and it's what actually resolves "profitable but too few trades":
   see the methodology cell below for why.
2. **Four independent grid-search experiments**, same fast<long lengths grid, same
   scoring, different indicator pair: KAMA(fast)/SMA(slow), EMA(fast)/SMA(slow),
   HMA(fast)/SMA(slow), DEMA(fast)/SMA(slow). Each prints a sorted-best-to-worst
   table and a heatmap of the ranking metric across the parameter grid.
3. **Cross-experiment comparison**: one table, best combo per indicator family.
4. **Out-of-sample check**: the winner evaluated on a held-out final slice of data it
   never touched during the grid search (the cheapest possible overfitting guard —
   see `OPTIMIZATION_RESEARCH_PLAN.md` §5.1).
5. **Final pick**, with honest caveats about what this pipeline does and doesn't
   establish.

**Real costs used below (researched 2026-07-10, see `PROP_FIRM_PLAN.md`)**:
- M6E (Micro EUR/USD) tick size: 0.0001 price units = $1.25/tick (CME contract spec).
- M6E commission: Tradeify's fee source confirms $1.82 round-turn for "micro contracts like MES and MNQ" but doesn't separately name M6E -- this assumes the same standard-micro rate (-> $0.91/side), not an M6E-specific citation (see `PROP_FIRM_PLAN.md`). This cost model
  charges per fill, matching `TradovateSymbolConfigProvider.get_fee()`'s per-side
  billing in the event-driven engine.
- Slippage: 1 tick/side (`TradovateSymbolConfigProvider`'s general default).
- These are no longer frictionless placeholders like the previous version of this
  notebook used — every number below already has trading costs baked in.

**Known gap carried over**: real M6E 1h data in Mongo only goes back to ~2025-07 even
though `DATE_FROM`/`DATE_TO` below span back to 2019 — so this is a search over ~1
year of data, not the ~6.7 years the bounds imply. See
`OPTIMIZATION_RESEARCH_PLAN.md` §2 for the data-backfill plan; nothing below should be
read as more statistically solid than "best available option given ~1 year of M6E
history," which is exactly why step 4 (out-of-sample check) matters.

**Deliberately out of scope for this pass** (see `OPTIMIZATION_RESEARCH_PLAN.md` for
why each is deferred, not forgotten):
- Re-tuning KAMA's own EMA fast/slow smoothing constants on top of the length grid
  (an earlier version of this notebook did this and found a modest improvement, but
  stacking a second grid search on the same ~1 year of data compounds overfitting
  risk for a gain that didn't survive this pass's out-of-sample check anyway).
- Breakout-style systems (Donchian, SuperTrend) — structurally not a two-line
  crossover, would need a new `Strategy` subclass in the event engine to use, not just
  a new indicator pair.
- Full walk-forward / CSCV overfitting-probability scoring, position sizing, and the
  Tradeify Monte-Carlo pass-probability simulator — all require either more data or
  more infrastructure than fits in this pass; tracked as `OPTIMIZATION_RESEARCH_PLAN.md`
  §2/§5.2-5.3/§7.2.


In [ ]:
import itertools as it
import os

import matplotlib.pyplot as plt
import multiprocess as mp
import numpy as np
import pandas as pd
import pandas_ta as ta
import seaborn as sns
from tqdm import tqdm

from backtester.exchange_config import TRADOVATE_FUTURES
from backtester.performance import metrics as perf_metrics
from data_aggregator.mongo_timescale_aggregator import fetch_market_data


In [ ]:
# prices is a pandas series with time as an index
######################################################################################################

DATE_FROM = "2019-10-01"
DATE_TO = "2026-06-01"

DISPLAY_SYMBOL = "M6E"    # bare symbol, as registered in Mongo `instruments`
SOURCE = "ib"             # Interactive Brokers continuous futures contract
RESOLUTION = "1_hour"     # Timescale candles are already pre-bucketed at this size

SIDE = 'long'

# --- Tradovate contract config: real, researched values (see intro + PROP_FIRM_PLAN.md) ---
instrument_config = TRADOVATE_FUTURES[DISPLAY_SYMBOL]
POINT_VALUE = instrument_config.point_value              # CME spec, see exchange_config.py
TICK_SIZE = instrument_config.tick_size                  # 0.0001 (CME spec)
SLIPPAGE_TICKS = instrument_config.slippage_ticks         # 1 tick/side (Tradovate provider default)
FEE_PER_CONTRACT = instrument_config.fee_per_contract_override  # see exchange_config.py

print(f"{DISPLAY_SYMBOL} cost model: tick_size={TICK_SIZE} (${TICK_SIZE*POINT_VALUE}/tick), "
      f"slippage={SLIPPAGE_TICKS} tick/side, commission=${FEE_PER_CONTRACT}/side")

######################################################################################################
cwd = os.getcwd()
print(cwd)

df = fetch_market_data(source=SOURCE, display_symbol=DISPLAY_SYMBOL, resolution=RESOLUTION)
df = df.loc[DATE_FROM:DATE_TO]
prices = df['close']

# Compute the log-returns of the underlying asset.
# https://investmentcache.com/magic-of-log-returns-concept-part-1/
rs = prices.apply(np.log).diff(1)

_span_days = (prices.index[-1] - prices.index[0]).days
print(f"Real data span: {prices.index[0]} -> {prices.index[-1]} "
      f"({_span_days} days, {len(prices)} bars) -- NOT the ~6.7 years DATE_FROM/DATE_TO imply, "
      f"see intro's 'Known gap' note.")


def backtest_side(val, side=SIDE):
    if side == 'both':
        return np.sign(val)
    if side == 'long':
        return 1 if val > 0 else 0
    if side == 'short':
        return -1 if val < 0 else 0
    raise ValueError(side)


def evaluate_signal(ma_x: pd.Series, eval_prices: pd.Series, eval_rs: pd.Series) -> dict:
    """Turns a fast-minus-slow signal series into the full risk-adjusted, cost-aware
    scorecard every experiment below is ranked on. `eval_prices`/`eval_rs` let the
    same function be reused on a data subset (e.g. the out-of-sample holdout below)
    without recomputing globals.

    Real Tradeify/Tradovate costs (1 tick slippage + $0.91/side commission) are
    applied on every position change, converted to a per-bar cost percentage exactly
    like the previous frictionless version of this notebook did -- the only change is
    the constants are no longer zero.
    """
    pos = ma_x.apply(backtest_side)
    pos_exec = pos.shift(1).fillna(0)

    dpos = pos_exec.diff().fillna(pos_exec)
    units_traded = dpos.abs()

    slippage_price_equivalent = SLIPPAGE_TICKS * TICK_SIZE
    fee_price_equivalent = FEE_PER_CONTRACT / POINT_VALUE
    cost_percent = (slippage_price_equivalent + fee_price_equivalent) / eval_prices
    cost_rs = units_traded * np.log1p(-cost_percent)

    net_rs = pos_exec * eval_rs + cost_rs
    total_return = float(np.exp(net_rs.sum()) - 1)

    # --- daily-resampled risk metrics: same convention PerformanceAnalyzer uses
    # (resample to daily, periods_per_year=365) so these numbers are methodologically
    # comparable to the event-driven engine's own report ---
    daily_log_returns = net_rs.resample("D").sum()
    daily_simple_returns = np.expm1(daily_log_returns)
    daily_cum = np.exp(daily_log_returns.cumsum())
    n_days = (daily_cum.index[-1] - daily_cum.index[0]).days or 1
    ann_return = perf_metrics.cagr(float(daily_cum.iloc[-1]), n_days)
    high_watermark = daily_cum.cummax()
    drawdown = (high_watermark - daily_cum) / high_watermark * -1
    max_dd = float(drawdown.min())

    sharpe = perf_metrics.sharpe_ratio(daily_simple_returns)
    sortino = perf_metrics.sortino_ratio(daily_simple_returns)
    calmar = perf_metrics.calmar_ratio(ann_return, max_dd)

    # --- trade-level Sharpe: n = number of ROUND-TRIP trades, not calendar days.
    # This -- not the daily-bar Sharpe above -- is what actually distinguishes "a
    # couple of lucky trades" from "a real, well-sampled edge": a daily-bar Sharpe
    # barely changes whether the window contains 30 trades or 300, since the number
    # of *days* in the backtest is the same either way. See metrics.sharpe_lower_bound
    # docstring. ---
    trade_id = (pos_exec != pos_exec.shift(1)).cumsum()
    in_trade = pos_exec != 0
    trade_log_returns = net_rs[in_trade].groupby(trade_id[in_trade]).sum()
    trade_returns = np.expm1(trade_log_returns)
    n_trades = len(trade_returns)
    trades_per_year = n_trades / (n_days / 365.25) if n_days else 0.0

    if n_trades >= 2:
        trade_sharpe = perf_metrics.sharpe_ratio(trade_returns, periods_per_year=trades_per_year)
        trade_sharpe_lb = perf_metrics.sharpe_lower_bound(trade_returns, periods_per_year=trades_per_year)
        win_rate = float((trade_returns > 0).mean())
    else:
        trade_sharpe = trade_sharpe_lb = win_rate = 0.0

    return {
        "total_return": total_return,
        "sharpe": sharpe,
        "sortino": sortino,
        "calmar": calmar,
        "max_drawdown": max_dd,
        # Tradeify Growth $50k account: $2,000 (4%) EOD-trailing drawdown (researched,
        # see PROP_FIRM_PLAN.md) -- this flags combos whose *frictionless-sizing*
        # equity curve already dipped this far, a cheap early warning even though real
        # dollar drawdown also depends on position sizing (only modeled properly in
        # the event-driven notebook).
        "breaches_tradeify_trail": max_dd < -0.04,
        "trades": n_trades,
        "trades_per_year": trades_per_year,
        "win_rate": win_rate,
        "trade_sharpe": trade_sharpe,
        "trade_sharpe_lb": trade_sharpe_lb,
    }


def run_grid(fast_fn, slow_fn, combinations, eval_prices=None, eval_rs=None) -> pd.DataFrame:
    """Runs evaluate_signal() over every (long_len, short_len) combo in parallel.
    fast_fn/slow_fn take a length and return that indicator's Series on eval_prices."""
    eval_prices = prices if eval_prices is None else eval_prices
    eval_rs = rs if eval_rs is None else eval_rs

    def _one(paramset):
        long_len, short_len = paramset
        ma_x = fast_fn(short_len) - slow_fn(long_len)
        result = evaluate_signal(ma_x, eval_prices, eval_rs)
        result["long_len"] = long_len
        result["short_len"] = short_len
        return result

    pool = mp.Pool(5, maxtasksperchild=50)
    results = list(tqdm(pool.imap(_one, combinations), total=len(combinations)))
    pool.close()
    return pd.DataFrame(results)


REPORT_COLUMNS = [
    "long_len", "short_len", "total_return", "trade_sharpe_lb", "trade_sharpe",
    "sharpe", "calmar", "max_drawdown", "trades", "trades_per_year", "win_rate",
    "breaches_tradeify_trail",
]
MIN_TRADES = 10  # sanity floor so a 2-3 trade fluke can't win on undefined/tiny-n stats


def report_experiment(name: str, results: pd.DataFrame) -> pd.DataFrame:
    """Filters to profitable combos with a sane minimum trade count, ranks by the
    trade-level Sharpe lower bound (best to worst), and returns/prints the table."""
    candidates = results[(results["total_return"] > 0) & (results["trades"] >= MIN_TRADES)]
    ranked = candidates.sort_values("trade_sharpe_lb", ascending=False)
    print(f"=== {name}: {len(ranked)}/{len(results)} combos profitable with >= {MIN_TRADES} trades ===")
    return ranked[REPORT_COLUMNS].head(15)


def plot_heatmap(name: str, results: pd.DataFrame, value_col: str = "trade_sharpe_lb"):
    """Heatmap of `value_col` over the (long_len, short_len) grid, plus a 3x3-smoothed
    'plateau' pick -- a parameter whose neighbors are also good is structurally less
    likely to be a one-cell overfit spike than the raw argmax."""
    pivot = results.pivot(index="long_len", columns="short_len", values=value_col)
    plt.figure(figsize=(14, 8))
    sns.heatmap(pivot, annot=False, cmap="viridis")
    plt.title(f"{name}: {value_col} across (long_len, short_len)")
    plt.show()

    smoothed = pivot.rolling(3, center=True, min_periods=1).mean()
    smoothed = smoothed.T.rolling(3, center=True, min_periods=1).mean().T
    plateau_long, plateau_short = smoothed.stack().idxmax()
    print(f"Plateau pick (3x3-smoothed argmax): long_len={plateau_long}, short_len={plateau_short}, "
          f"smoothed {value_col}={smoothed.loc[plateau_long, plateau_short]:.4f}")


### Bug found (while this was still SMA-based): the original grid conflated "long"/"short" labels with which *range* a value came from, not its actual size

The original grid's `long_ma_length = range(2, 1000, 10)` and `short_ma_length =
range(2, 300, 5)` **overlapped** (both started at 2). It took every `(long, short)`
pair from their Cartesian product and labeled them by *which range they came from* --
so a value drawn from the "short" range could easily be numerically **larger** than a
value drawn from the "long" range, producing a row labeled e.g. `long_ma=42,
short_ma=52`.

That's not just a cosmetic mislabel: the backtest function computes `signal(short) -
signal(long)`. When `short > long`, this silently becomes **slow − fast** instead of
the intended fast − slow, flipping the crossover rule from trend-following into
mean-reversion while still being labeled as if it were the trend-following rule with a
nonsensical window pairing.

Checked directly against the real MES data (SMA version): combos where `short_ma >
long_ma` averaged **12.6%** return vs. **4.8%** for correctly-ordered pairs -- exactly
the "hotter when short > long" pattern that got caught in the original heatmap.

**Fix, applied to every experiment below**: filter to only keep `short_len < long_len`
pairs (the `combinations` list below), so every row is an actual, correctly-labeled
crossover -- shared across all four indicator-pair experiments so they're all searched
over identical parameter space.

### Methodology: why rank by a trade-level Sharpe *lower bound* instead of raw return or a `MIN_TRADES` cutoff

Sorted purely by return, the top row of any of these grids tends to be a combo with
very few trades -- e.g. an earlier version of this search surfaced
`long_len=132, short_len=46` at +15.5% frictionless, but only **33 trades** over the
~1 year of MES data actually available. A couple of lucky or unlucky fills can swing a
33-trade result by several points, so a raw-return leaderboard systematically rewards
noise.

A hard `trades >= N` cutoff (what the previous version of this notebook did) is a step
better but still arbitrary -- it can't say *how much* to trust a 40-trade result vs. a
140-trade one, just whether they clear one line.

**What's used below instead**: `metrics.sharpe_lower_bound()` (Lo 2002's asymptotic
Sharpe standard error, `SE(SR) ~= sqrt((1 + SR^2/2) / n)`), computed on **per-trade**
returns with `n` = number of round-trip trades. This directly encodes "how much do we
trust this Sharpe estimate" as a function of sample size: two combos with the same
point-estimate Sharpe are *not* scored the same if one has 4x the trades -- the
lower-trade one gets a bigger penalty subtracted off. This is exactly the "profitable
but don't want just a few trades" preference, expressed as math instead of a threshold.
`MIN_TRADES = 10` below is kept only as a sanity floor (avoids a 2-3-trade combo
winning on a degenerate/undefined tiny-sample statistic), not as the selection
mechanism itself.

Each experiment cell below also prints a **plateau pick**: the parameter whose 3x3
neighborhood (in the long_len x short_len grid) has the best *average* score, not just
the single best cell. A pick surrounded by other good picks is structurally less
likely to be a one-cell overfit spike.

In [ ]:
# shared parameter grid across all four experiments below -- same lookback ranges
# used by the original KAMA-only search, so results are comparable across indicator
# families, not just across lengths within one family.
long_len_range = range(2, 300, 10)
short_len_range = range(2, 60, 4)
combinations = [c for c in it.product(long_len_range, short_len_range) if c[1] < c[0]]
print(f"{len(combinations)} valid (long_len > short_len) combinations per experiment")


## Experiment 1: KAMA(fast) / SMA(slow)

Kaufman's Adaptive Moving Average for the fast line (adapts its smoothing to how
directional vs. choppy recent price action has been), plain SMA for the slow line.
KAMA's own `fast`/`slow` EMA constants are held at Kaufman's defaults (2, 30) -- not
re-tuned this pass, see intro.

In [ ]:
kama_sma_fast_fn = lambda w: ta.kama(prices, length=w, fast=2, slow=30)  # noqa: E731
kama_sma_slow_fn = lambda w: prices.rolling(w).mean()  # noqa: E731

kama_sma_results = run_grid(kama_sma_fast_fn, kama_sma_slow_fn, combinations)
report_experiment("KAMA/SMA", kama_sma_results)


In [ ]:
plot_heatmap("KAMA/SMA", kama_sma_results)


## Experiment 2: EMA(fast) / SMA(slow)

Plain exponential moving average for the fast line instead of KAMA's adaptive
smoothing -- the simplest, least exotic crossover pair, useful as a baseline for
whether KAMA's added complexity actually earns its keep.

In [ ]:
ema_sma_fast_fn = lambda w: ta.ema(prices, length=w)  # noqa: E731
ema_sma_slow_fn = lambda w: prices.rolling(w).mean()  # noqa: E731

ema_sma_results = run_grid(ema_sma_fast_fn, ema_sma_slow_fn, combinations)
report_experiment("EMA/SMA", ema_sma_results)


In [ ]:
plot_heatmap("EMA/SMA", ema_sma_results)


## Experiment 3: HMA(fast) / SMA(slow)

Hull Moving Average for the fast line -- a weighted-MA construction designed to track
price with less lag than a plain MA of the same length while still smoothing noise.

In [ ]:
hma_sma_fast_fn = lambda w: ta.hma(prices, length=w)  # noqa: E731
hma_sma_slow_fn = lambda w: prices.rolling(w).mean()  # noqa: E731

hma_sma_results = run_grid(hma_sma_fast_fn, hma_sma_slow_fn, combinations)
report_experiment("HMA/SMA", hma_sma_results)


In [ ]:
plot_heatmap("HMA/SMA", hma_sma_results)


## Experiment 4: DEMA(fast) / SMA(slow)

Double Exponential Moving Average for the fast line -- applies EMA twice and combines
them (2*EMA - EMA(EMA)) to cancel out most of a plain EMA's lag at the same length.

In [ ]:
dema_sma_fast_fn = lambda w: ta.dema(prices, length=w)  # noqa: E731
dema_sma_slow_fn = lambda w: prices.rolling(w).mean()  # noqa: E731

dema_sma_results = run_grid(dema_sma_fast_fn, dema_sma_slow_fn, combinations)
report_experiment("DEMA/SMA", dema_sma_results)


In [ ]:
plot_heatmap("DEMA/SMA", dema_sma_results)


## Cross-experiment comparison

Best (highest `trade_sharpe_lb`) combo from each of the four experiments above, side
by side.

In [ ]:
all_experiments = {
    "KAMA/SMA": kama_sma_results,
    "EMA/SMA": ema_sma_results,
    "HMA/SMA": hma_sma_results,
    "DEMA/SMA": dema_sma_results,
}

best_per_experiment = []
for name, results in all_experiments.items():
    candidates = results[(results["total_return"] > 0) & (results["trades"] >= MIN_TRADES)]
    if candidates.empty:
        continue
    best = candidates.sort_values("trade_sharpe_lb", ascending=False).iloc[0].to_dict()
    best["experiment"] = name
    best_per_experiment.append(best)

comparison = pd.DataFrame(best_per_experiment).sort_values("trade_sharpe_lb", ascending=False)
comparison[["experiment", *REPORT_COLUMNS]]


## Out-of-sample check: does the winner hold up on data it never touched?

The cheapest possible overfitting guard (`OPTIMIZATION_RESEARCH_PLAN.md` §5.1): split
the ~1 year of data into the first 75% ("train", what every grid above searched over)
and the last 25% ("test", untouched until now), then re-evaluate the overall winner's
exact parameters on each slice plus the full window. If the strategy's edge is real
rather than a fit to the specific bars in the training slice, performance and
trade-level Sharpe shouldn't collapse on the held-out slice.

In [ ]:
# fast/slow line factories, keyed the same way as `all_experiments` above, so the
# winner (whichever experiment it comes from) can be re-evaluated on data subsets.
fast_fns = {
    "KAMA/SMA": kama_sma_fast_fn,
    "EMA/SMA": ema_sma_fast_fn,
    "HMA/SMA": hma_sma_fast_fn,
    "DEMA/SMA": dema_sma_fast_fn,
}
slow_fns = {
    "KAMA/SMA": kama_sma_slow_fn,
    "EMA/SMA": ema_sma_slow_fn,
    "HMA/SMA": hma_sma_slow_fn,
    "DEMA/SMA": dema_sma_slow_fn,
}

winner = comparison.iloc[0]
winner_experiment = winner["experiment"]
winner_long, winner_short = int(winner["long_len"]), int(winner["short_len"])
print(f"Overall winner: {winner_experiment}  long_len={winner_long}  short_len={winner_short}")

split = int(len(prices) * 0.75)
split_date = prices.index[split]
print(f"train/test split at {split_date} ({split} train bars, {len(prices) - split} test bars)\n")

holdout_rows = []
for label, sub_prices in [
    ("FULL (searched over)", prices),
    ("TRAIN (first 75%)", prices.iloc[:split]),
    ("TEST (last 25%, unseen by the grid)", prices.iloc[split:]),
]:
    sub_rs = sub_prices.apply(np.log).diff(1)
    fast = fast_fns[winner_experiment](winner_short)
    slow = slow_fns[winner_experiment](winner_long)
    # re-slice fast/slow to this subset's index (they were computed against the full
    # `prices` series above so each MA's warmup window still sees its true history)
    ma_x = (fast - slow).loc[sub_prices.index]
    row = evaluate_signal(ma_x, sub_prices, sub_rs)
    row["window"] = label
    holdout_rows.append(row)

holdout_df = pd.DataFrame(holdout_rows).set_index("window")
holdout_df[["total_return", "trades", "trade_sharpe", "trade_sharpe_lb", "max_drawdown", "breaches_tradeify_trail"]]


## Final pick and honest caveats

**Result: weaker than MES/M2K/MNQ, and worth being upfront about.** For M6E, **all
four indicator families have a negative `trade_sharpe_lb`** -- there is no
crossover-family combo in this search that clears zero confidently. The best
available is HMA(fast)/SMA(slow), `long_len=62`, `short_len=58`: +1.6% over **46
round-trip trades** (~51/year), `trade_sharpe_lb` of **-0.49** (next-best: EMA/SMA at
-0.56, DEMA/SMA at -0.57, KAMA/SMA worst at -0.66). This is "least bad," not "good" --
unlike MES (+0.04), M2K (+0.10), and MNQ (+0.04), which all had at least one family
clear zero.

**The out-of-sample check underlines this**: the test slice is actually *negative*
total return (-1.1%) with a deeply negative `trade_sharpe_lb` (-3.25) on just 15
trades. There's no basis here to claim this crossover strategy has a real edge on
M6E over this window -- it's included in the combined-portfolio selection process
anyway because M6E's *diversification* value (near-zero correlation with the equity
index book, see `ib_portfolio_test.ipynb`) is a separate, real property from whether
this particular trend-following strategy makes money on it standalone. Whether that
diversification benefit is worth carrying a weak/flat standalone strategy is exactly
what the combined portfolio's 3-of-6 selection step should reveal.

**Drawdown is genuinely small** here (max ~2.5-3.7% across full/train/test, never
breaching the Tradeify trail) -- consistent with FX generally being lower-volatility
than equity index futures at the same notional.

**Next**: this parameter set (HMA(58)/SMA(62)) is implemented as a candidate strategy
in `step_2_run_full_backtest.ipynb` and in the combined portfolio's 3-of-6 selection
-- but going in with the expectation that it may not make the final cut, given the
weak standalone numbers above.